# SEIRD Model: Adding Deceased Compartment and Generalized Compartment Handling

**Navigation:**  [← SEIRS: Waning Immunity](03_seirs.ipynb) | [Two-Population Structure →](05_two_population.ipynb)


This notebook demonstrates the SEIRD model, which extends SEIR by adding a Deceased (D) compartment. It also introduces a generalized approach to compartment modeling, paving the way for flexible future extensions.

In [ ]:
# 1. Load Repository Metadata and List Notebook Files
import os
import subprocess
from pathlib import Path
import re

NOTEBOOK_DIR = Path("../notebooks").resolve()
assert NOTEBOOK_DIR.exists(), f"Notebook directory {NOTEBOOK_DIR} does not exist"

# Ensure we're on main branch
git_branch = subprocess.check_output(["git", "branch", "--show-current"]).decode().strip()
assert git_branch == "main", f"Not on main branch: {git_branch}"

# List all numbered notebook files
notebook_files = sorted([f for f in os.listdir(NOTEBOOK_DIR) if re.match(r"\\d{2}_.*\\.ipynb", f)])
print("Current notebook files:")
for f in notebook_files:
    print(f)
assert notebook_files, "No numbered notebooks found!"

## 2. Identify Insertion Point After the SEIRS Notebook

We parse notebook filenames and titles, locate the notebook matching SEIRS, and compute the target index for the new SEIRD notebook.

In [ ]:
# 2. Identify Insertion Point After the SEIRS Notebook
seirs_pattern = re.compile(r"\\d{2}_seirs\\.ipynb", re.IGNORECASE)
seirs_idx = None
for idx, fname in enumerate(notebook_files):
    if seirs_pattern.match(fname):
        seirs_idx = idx
        break
assert seirs_idx is not None, "Could not find SEIRS notebook!"
print(f"SEIRS notebook found at index {seirs_idx}: {notebook_files[seirs_idx]}")
insert_idx = seirs_idx + 1
print(f"New SEIRD notebook will be inserted at index {insert_idx}")

## 3. Build a Collision-Free Renaming Plan

We generate a deterministic rename map that shifts later notebook numbers up by 1. Reverse-order renaming avoids filename collisions without temp files.

In [ ]:
# 3. Build a Collision-Free Renaming Plan
rename_plan = []
for i in range(len(notebook_files) - 1, insert_idx - 1, -1):
    old_name = notebook_files[i]
    old_num = int(old_name[:2])
    new_num = old_num + 1
    new_name = f"{new_num:02d}_" + old_name[3:]
    rename_plan.append((old_name, new_name))

print("Planned renames (reverse order):")
for old, new in rename_plan:
    print(f"{old} -> {new}")

print("\nReverse-order renaming avoids collisions because each file is moved to a new, unused name before the next.")

## 4. Dry-Run the `git mv` Operations

We print the exact shell commands for each rename in reverse numeric order, and show before/after paths for verification.

In [ ]:
# 4. Dry-Run the `git mv` Operations
for old, new in rename_plan:
    print(f"git mv '{NOTEBOOK_DIR / old}' '{NOTEBOOK_DIR / new}'")
print("\nDry-run complete. No files have been moved yet.")

## 5. Execute In-Place Renames in Reverse Order

We run the prepared `git mv` commands from highest index to lowest shifted index, capturing command output and failing fast on any error.

In [ ]:
# 5. Execute In-Place Renames in Reverse Order
for old, new in rename_plan:
    cmd = ["git", "mv", str(NOTEBOOK_DIR / old), str(NOTEBOOK_DIR / new)]
    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True)
    if result.returncode != 0:
        print(result.stderr.decode())
        raise RuntimeError(f"Failed to rename {old} to {new}")
print("All renames completed.")

## 6. Insert the New Notebook with the Correct Number

We create the new notebook filename at the computed slot, scaffold initial cells programmatically, and ensure naming consistency.

In [ ]:
# 6. Insert the New Notebook with the Correct Number
new_num = int(notebook_files[seirs_idx][:2]) + 1
new_nb_name = f"{new_num:02d}_seird.ipynb"
new_nb_path = NOTEBOOK_DIR / new_nb_name

# Scaffold a minimal notebook if not present
if not new_nb_path.exists():
    import nbformat
    nb = nbformat.v4.new_notebook()
    nb.cells.append(nbformat.v4.new_markdown_cell("# SEIRD Model\n\nThis notebook demonstrates SEIRD modeling with a generalized compartment list."))
    with open(new_nb_path, "w") as f:
        nbformat.write(nb, f)
    print(f"Created new notebook: {new_nb_path}")
else:
    print(f"Notebook already exists: {new_nb_path}")

## 7. Validate Ordering, Git Status, and Notebook Execution

We re-list notebook files to confirm contiguous numbering, run a quick execution check with nbclient or papermill, and show `git status` to verify only intended renames/additions.

In [ ]:
# 7. Validate Ordering, Git Status, and Notebook Execution
notebook_files_post = sorted([f for f in os.listdir(NOTEBOOK_DIR) if re.match(r"\\d{2}_.*\\.ipynb", f)])
print("Notebook files after insertion:")
for f in notebook_files_post:
    print(f)

# Check for contiguous numbering
nums = [int(f[:2]) for f in notebook_files_post]
assert nums == list(range(1, max(nums) + 1)), f"Notebook numbers not contiguous: {nums}"

# Show git status
print("\nGit status:")
print(subprocess.check_output(["git", "status", "--short", str(NOTEBOOK_DIR)]).decode())

# Quick execution check (optional, requires nbclient)
try:
    import nbclient
    from nbclient import NotebookClient
    for f in notebook_files_post:
        nb_path = NOTEBOOK_DIR / f
        print(f"Executing {nb_path} ...")
        nb = nbformat.read(open(nb_path), as_version=4)
        client = NotebookClient(nb)
        client.execute()
    print("All notebooks executed successfully.")
except ImportError:
    print("nbclient not installed; skipping execution check.")